In [0]:
data_day1 = [
    (1, "C001", "Laptop", 50000, "2024-01-01"),
    (2, "C002", "Mobile", 15000, "2024-01-02"),
    (3, "C003", "Tablet", 20000, "2024-01-03"),
    (4, "C004", "Laptop", 55000, "2024-01-04"),
    (5, "C005", "Headphones", 3000, "2024-01-05")
]

columns = ["order_id", "customer_id", "product", "amount", "updated_date"]

df_day1 = spark.createDataFrame(data_day1, columns)
df_day1.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,15000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,3000,2024-01-05


In [0]:
%sql
CREATE VOLUME workspace.default.practice;

In [0]:
df_day1.write \
    .mode("overwrite") \
    .parquet("/Volumes/workspace/default/practice")

In [0]:
existing_df = spark.read.parquet("/Volumes/workspace/default/practice")
existing_df.display()

order_id,customer_id,product,amount,updated_date
5,C005,Headphones,3000,2024-01-05
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,15000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04


In [0]:
from pyspark.sql.functions import *
last_date=existing_df.select(max("updated_date")).collect()[0][0]
print(last_date)

2024-01-05


In [0]:
data_day2 = [
    (3, "C003", "Tablet", 22000, "2024-01-06"),  # updated
    (4, "C004", "Laptop", 56000, "2024-01-06"),  # updated
    (6, "C006", "Camera", 30000, "2024-01-06"),  # new
    (7, "C007", "Mobile", 18000, "2024-01-07"),  # new
    (8, "C008", "Watch", 8000, "2024-01-02")     # new
]

df_day2 = spark.createDataFrame(data_day2, columns)
df_day2.display()
new_df = df_day2

order_id,customer_id,product,amount,updated_date
3,C003,Tablet,22000,2024-01-06
4,C004,Laptop,56000,2024-01-06
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-02


In [0]:
from pyspark.sql.functions import col

incremental_df = new_df.filter(
    col("updated_date") > last_date
)
incremental_df.display()

order_id,customer_id,product,amount,updated_date
3,C003,Tablet,22000,2024-01-06
4,C004,Laptop,56000,2024-01-06
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07


In [0]:
filtered_existing = existing_df.join(
    incremental_df,
    on="order_id",
    how="left_anti"
)
filtered_existing.display()

order_id,customer_id,product,amount,updated_date
5,C005,Headphones,3000,2024-01-05
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,15000,2024-01-02


In [0]:
a=[
    (1,2),
    (3,4),
    (5,6)
]
columns=["id","sal"]
df1=spark.createDataFrame(a,columns)
b=[
    (1,2),
    (3,4),
    (6,5)
]
col=["id","sal1"]
df2=spark.createDataFrame(b,col)
res1=df2.union(df1)
res2=df1.join(df2,"id","inner")
res1.display()
res2.display()

id,sal1
1,2
3,4
6,5
1,2
3,4
5,6


id,sal,sal1
1,2,2
3,4,4
